# 화요일 단독 모델 실증 — 스텝와이즈 OLS (T1 · T2)

`02_feature_correlation_dilution.ipynb`에서 확인한 상관계수 희석 문제를, 화요일 단독 표본(7~8행)으로 실제 회귀모델을 쌓아가며 검증한다. T1(`cumul_16_19`)과 T2(`cumul_20_22`) 두 타겟에 동일한 절차(상관계수 확인 → 후보 추출 → 단순회귀 → 스텝와이즈)를 반복하므로, 절차 자체를 함수로 뽑아 재사용한다.


## 설정 및 데이터 준비


In [1]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

data = pd.read_csv('/content/time_table.csv')

tue_data = data[data['day_of_week'] == '화']  # 연속성 유지 요일
tue_data_not_outlier = tue_data[tue_data['date'] != '2026-06-30']  # 전날 이상치로 인한 변동성 확대분 제외

print(f"화요일 표본: 이상치 포함 {len(tue_data)}행 / 이상치 미포함 {len(tue_data_not_outlier)}행")


화요일 표본: 이상치 포함 8행 / 이상치 미포함 7행


In [2]:
time_list = [
    'cumul_16_19', 'cumul_20_22',
    'lag_prev_day_t1', 'lag_same_day_t1', 'lag_prev_day_t2', 'lag_same_day_t2',
    'lag_biweekly_t1', 'lag_biweekly_t2',
    'roll_prev_3days_mean_t1', 'roll_same_3days_mean_t1',
    'roll_prev_3days_mean_t2', 'roll_same_3days_mean_t2',
    'roll_prev_5days_mean_t1', 'roll_prev_5days_mean_t2',
    'lag_prev_t1_ratio', 'lag_same_t1_ratio', 'lag_prev_t2_ratio', 'lag_same_t2_ratio',
    'prev_sum_g', 'same_prev_sum_g',
]


## 재사용 함수

아래 4개 함수가 T1·T2 양쪽에서 반복되는 절차를 담당한다.
1. `corr_report` — 이상치 포함/미포함 상관계수 비교 출력
2. `select_candidates` — 상관계수 절댓값 기준 후보 피처 추출
3. `single_feature_scan` — 후보 피처별 단순회귀 R²/Adj R²/p값 비교
4. `stepwise_scan` — 이미 선택된 피처에 남은 후보를 하나씩 추가했을 때의 개선 여부 비교


In [3]:
def corr_report(df_incl, df_excl, target, cols):
    print(f"이상치 포함 화요일 시계열 상관계수 ({target})")
    print(df_incl[cols].corr()[target].sort_values(ascending=False))
    print()
    print(f"이상치 미포함 화요일 시계열 상관계수 ({target})")
    print(df_excl[cols].corr()[target].sort_values(ascending=False))


def select_candidates(df, target, cols, threshold=0.3):
    """상관계수 절댓값이 threshold 이상인 피처만 골라 반환한다 (target 자기 자신은 제외)."""
    corr = df[cols].corr()[target].abs()
    return [c for c in corr[corr > threshold].index.tolist() if c != target]


def single_feature_scan(df, features, target, label):
    print(f"{label} 단순회귀 계수확인")
    for feature in features:
        subset = df[[feature, target]].dropna()
        X = sm.add_constant(subset[[feature]])
        y = subset[target]
        model = sm.OLS(y, X).fit()
        print(f"{feature:<30} R²={model.rsquared:.3f}  Adj R²={model.rsquared_adj:.3f}  p={model.pvalues[feature]:.3f}")


def stepwise_scan(df, selected, remaining, target, label):
    print(f"【{label}】 현재 피처: {selected}")
    print(f"{'피처 추가':<30} {'R²':<8} {'Adj R²':<10} {'p(추가피처)'}")
    print('-' * 65)
    for feat in remaining:
        cols = selected + [feat]
        subset = df[cols + [target]].dropna()
        X = sm.add_constant(subset[cols])
        y = subset[target]
        model = sm.OLS(y, X).fit()
        print(f"{feat:<30} R²={model.rsquared:.3f}  Adj R²={model.rsquared_adj:.3f}  p={model.pvalues[feat]:.3f}")
    print()


def fit_ols(df, cols, target):
    subset = df[cols + [target]].dropna()
    X = sm.add_constant(subset[cols])
    y = subset[target]
    return sm.OLS(y, X).fit()


---
# 1. 타겟: cumul_16_19 (T1)


## 1-1. 상관계수 — 이상치 포함/미포함 비교


In [4]:
corr_report(tue_data, tue_data_not_outlier, 'cumul_16_19', time_list)


이상치 포함 화요일 시계열 상관계수 (cumul_16_19)
cumul_16_19                1.000000
cumul_20_22                0.427325
roll_same_3days_mean_t2    0.422212
roll_prev_5days_mean_t1    0.175762
roll_prev_3days_mean_t2    0.155076
lag_prev_day_t1            0.138239
lag_same_t2_ratio          0.130990
prev_sum_g                 0.073167
lag_prev_t1_ratio          0.045103
roll_prev_5days_mean_t2    0.035027
lag_same_day_t2            0.011682
lag_prev_day_t2           -0.010448
lag_prev_t2_ratio         -0.045103
lag_same_t1_ratio         -0.130990
roll_prev_3days_mean_t1   -0.139800
same_prev_sum_g           -0.177869
lag_biweekly_t2           -0.234955
lag_same_day_t1           -0.281477
lag_biweekly_t1           -0.674892
roll_same_3days_mean_t1   -0.799705
Name: cumul_16_19, dtype: float64

이상치 미포함 화요일 시계열 상관계수 (cumul_16_19)
cumul_16_19                1.000000
roll_prev_5days_mean_t1    0.565282
lag_prev_day_t1            0.459259
cumul_20_22                0.393298
lag_same_t1_ratio          0.348

### 화요일 T1(16_19) 유효 값 후보 (절댓값 0.3 이상)
1. roll_prev_5days_mean_t1  0.565  (spurious 주의)
2. lag_biweekly_t1         -0.579  (격주 오후)
3. lag_prev_day_t1          0.459  (직전 영업일 오후)
4. lag_same_t1_ratio        0.348  (동요일 직전 오후 비중)
5. lag_same_t2_ratio       -0.348  (동요일 직전 저녁 비중)
6. roll_prev_3days_mean_t1  0.320
7. roll_same_3days_mean_t2  0.318  (spurious 주의)
8. prev_sum_g               0.303  (직전 영업일 전체 수요)
9. roll_same_3days_mean_t1 -0.755  (spurious 주의)


## 1-2. 후보 피처 추출 (|상관계수| > 0.3)


In [5]:
features_excl = select_candidates(tue_data_not_outlier, 'cumul_16_19', time_list)
features_incl = select_candidates(tue_data, 'cumul_16_19', time_list)
print("이상치 미포함 피처 후보:", features_excl)
print("이상치 포함 피처 후보:", features_incl)


이상치 미포함 피처 후보: ['cumul_20_22', 'lag_prev_day_t1', 'lag_biweekly_t1', 'roll_prev_3days_mean_t1', 'roll_same_3days_mean_t1', 'roll_same_3days_mean_t2', 'roll_prev_5days_mean_t1', 'lag_same_t1_ratio', 'lag_same_t2_ratio', 'prev_sum_g']
이상치 포함 피처 후보: ['cumul_20_22', 'lag_biweekly_t1', 'roll_same_3days_mean_t1', 'roll_same_3days_mean_t2']


## 1-3. 단순회귀 — 후보 피처별 단독 설명력


In [6]:
single_feature_scan(tue_data_not_outlier, features_excl, 'cumul_16_19', '이상치 미포함')


이상치 미포함 단순회귀 계수확인
cumul_20_22                    R²=0.155  Adj R²=-0.014  p=0.383
lag_prev_day_t1                R²=0.211  Adj R²=0.053  p=0.300
lag_biweekly_t1                R²=0.336  Adj R²=0.114  p=0.306
roll_prev_3days_mean_t1        R²=0.103  Adj R²=-0.122  p=0.536
roll_same_3days_mean_t1        R²=0.570  Adj R²=0.355  p=0.245
roll_same_3days_mean_t2        R²=0.101  Adj R²=-0.348  p=0.682
roll_prev_5days_mean_t1        R²=0.320  Adj R²=0.093  p=0.321
lag_same_t1_ratio              R²=0.121  Adj R²=-0.098  p=0.499
lag_same_t2_ratio              R²=0.121  Adj R²=-0.098  p=0.499
prev_sum_g                     R²=0.092  Adj R²=-0.090  p=0.508


In [7]:
single_feature_scan(tue_data, features_incl, 'cumul_16_19', '이상치 포함')


이상치 포함 단순회귀 계수확인
cumul_20_22                    R²=0.183  Adj R²=0.046  p=0.291
lag_biweekly_t1                R²=0.455  Adj R²=0.319  p=0.141
roll_same_3days_mean_t1        R²=0.640  Adj R²=0.519  p=0.104
roll_same_3days_mean_t2        R²=0.178  Adj R²=-0.096  p=0.479


## 이상치 포함 데이터에서 설명력이 더 높게 보이지만
## 소표본 환경에서 이상치 1개의 영향력이 크므로
## 이상치 미포함 모델 기준으로 피처 선별 진행
## (이상치 포함 시 왜곡 패턴 가능성 있음)


## 1-4. 스텝와이즈 전진선택


In [8]:
select_feature = ['lag_biweekly_t1']  # 가장 높은 설명력 피처
remain_features = [f for f in features_excl if f not in select_feature]
stepwise_scan(tue_data_not_outlier, select_feature, remain_features, 'cumul_16_19', 'Step 1 — lag_biweekly_t1 기준')


【Step 1 — lag_biweekly_t1 기준】 현재 피처: ['lag_biweekly_t1']
피처 추가                          R²       Adj R²     p(추가피처)
-----------------------------------------------------------------
cumul_20_22                    R²=0.337  Adj R²=-0.325  p=0.948
lag_prev_day_t1                R²=0.969  Adj R²=0.937  p=0.024
roll_prev_3days_mean_t1        R²=0.966  Adj R²=0.933  p=0.026
roll_same_3days_mean_t1        R²=0.876  Adj R²=0.627  p=0.316
roll_same_3days_mean_t2        R²=0.452  Adj R²=-0.643  p=0.957
roll_prev_5days_mean_t1        R²=0.530  Adj R²=0.060  p=0.459
lag_same_t1_ratio              R²=0.436  Adj R²=-0.129  p=0.612
lag_same_t2_ratio              R²=0.436  Adj R²=-0.129  p=0.612
prev_sum_g                     R²=0.971  Adj R²=0.942  p=0.022



2번째 피처 후보 3개(Adj R² 상위) 각각을 기준으로 다음 단계 후보를 비교한다: **A안** `prev_sum_g`(0.942) / **B안** `lag_prev_day_t1`(0.937) / **C안** `roll_prev_3days_mean_t1`(0.933)


In [9]:
all_features = [f for f in features_excl if f != 'lag_biweekly_t1']

for label, second_feat in [
    ('Step 2-A (prev_sum_g)', 'prev_sum_g'),
    ('Step 2-B (lag_prev_day_t1)', 'lag_prev_day_t1'),
    ('Step 2-C (roll_prev_3days_mean_t1)', 'roll_prev_3days_mean_t1'),
]:
    selected = ['lag_biweekly_t1', second_feat]
    remaining = [f for f in all_features if f != second_feat]
    stepwise_scan(tue_data_not_outlier, selected, remaining, 'cumul_16_19', label)


【Step 2-A (prev_sum_g)】 현재 피처: ['lag_biweekly_t1', 'prev_sum_g']
피처 추가                          R²       Adj R²     p(추가피처)
-----------------------------------------------------------------
cumul_20_22                    R²=0.996  Adj R²=0.984  p=0.243
lag_prev_day_t1                R²=0.978  Adj R²=0.912  p=0.669
roll_prev_3days_mean_t1        R²=0.987  Adj R²=0.947  p=0.471
roll_same_3days_mean_t1        R²=1.000  Adj R²=nan  p=nan
roll_same_3days_mean_t2        R²=1.000  Adj R²=nan  p=nan
roll_prev_5days_mean_t1        R²=0.996  Adj R²=0.984  p=0.245
lag_same_t1_ratio              R²=0.978  Adj R²=0.911  p=0.676
lag_same_t2_ratio              R²=0.978  Adj R²=0.911  p=0.676

【Step 2-B (lag_prev_day_t1)】 현재 피처: ['lag_biweekly_t1', 'lag_prev_day_t1']
피처 추가                          R²       Adj R²     p(추가피처)
-----------------------------------------------------------------
cumul_20_22                    R²=0.969  Adj R²=0.875  p=0.949
roll_prev_3days_mean_t1        R²=0.971  Adj R²=0.

3번째 피처를 추가하는 모든 분기에서 R²=1.000·Adj R²=nan(완전포화) 사례가 즉시 나타난다 — 표본(5~6행)에 비해 피처가 많아지며 수학적으로 무의미해진 상태다. 따라서 2피처(`lag_biweekly_t1` + `prev_sum_g`, A안 기준 Adj R² 최고)에서 스텝와이즈를 멈춘다.


## 1-5. 최종 모델 — cumul_16_19


In [10]:
final_model_t1 = fit_ols(tue_data_not_outlier, ['lag_biweekly_t1', 'prev_sum_g'], 'cumul_16_19')
print("【최종 모델 - 화요일 cumul_16_19】")
print(final_model_t1.summary())


【최종 모델 - 화요일 cumul_16_19】
                            OLS Regression Results                            
Dep. Variable:            cumul_16_19   R-squared:                       0.971
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     33.37
Date:                Tue, 14 Jul 2026   Prob (F-statistic):             0.0291
Time:                        05:10:45   Log-Likelihood:                -37.735
No. Observations:                   5   AIC:                             81.47
Df Residuals:                       2   BIC:                             80.30
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const           

---
# 2. 타겟: cumul_20_22 (T2)


## 2-1. 상관계수 — 이상치 포함/미포함 비교


In [11]:
corr_report(tue_data, tue_data_not_outlier, 'cumul_20_22', time_list)


이상치 포함 화요일 시계열 상관계수 (cumul_20_22)
cumul_20_22                1.000000
lag_same_t2_ratio          0.597001
cumul_16_19                0.427325
lag_same_day_t2            0.304386
lag_prev_t2_ratio          0.171366
lag_prev_day_t2            0.104388
prev_sum_g                 0.021184
lag_biweekly_t1            0.007578
roll_prev_3days_mean_t2   -0.006489
same_prev_sum_g           -0.061342
lag_prev_day_t1           -0.080800
lag_prev_t1_ratio         -0.171366
roll_same_3days_mean_t2   -0.242240
roll_prev_3days_mean_t1   -0.276674
lag_same_day_t1           -0.345570
roll_same_3days_mean_t1   -0.352975
lag_biweekly_t2           -0.381120
lag_same_t1_ratio         -0.597001
roll_prev_5days_mean_t2   -0.691005
roll_prev_5days_mean_t1   -0.899024
Name: cumul_20_22, dtype: float64

이상치 미포함 화요일 시계열 상관계수 (cumul_20_22)
cumul_20_22                1.000000
lag_same_t2_ratio          0.444526
cumul_16_19                0.393298
lag_biweekly_t1            0.374505
prev_sum_g                 0.227

## 2-2. 후보 피처 추출 (|상관계수| > 0.3)


In [12]:
features_excl_t2 = select_candidates(tue_data_not_outlier, 'cumul_20_22', time_list)
features_incl_t2 = select_candidates(tue_data, 'cumul_20_22', time_list)
print("이상치 미포함 피처 후보:", features_excl_t2)
print("이상치 포함 피처 후보:", features_incl_t2)


이상치 미포함 피처 후보: ['cumul_16_19', 'lag_biweekly_t1', 'lag_biweekly_t2', 'roll_same_3days_mean_t2', 'roll_prev_5days_mean_t1', 'roll_prev_5days_mean_t2', 'lag_same_t1_ratio', 'lag_same_t2_ratio']
이상치 포함 피처 후보: ['cumul_16_19', 'lag_same_day_t1', 'lag_same_day_t2', 'lag_biweekly_t2', 'roll_same_3days_mean_t1', 'roll_prev_5days_mean_t1', 'roll_prev_5days_mean_t2', 'lag_same_t1_ratio', 'lag_same_t2_ratio']


## 2-3. 단순회귀 — 후보 피처별 단독 설명력


In [13]:
single_feature_scan(tue_data_not_outlier, features_excl_t2, 'cumul_20_22', '이상치 미포함')


이상치 미포함 단순회귀 계수확인
cumul_16_19                    R²=0.155  Adj R²=-0.014  p=0.383
lag_biweekly_t1                R²=0.140  Adj R²=-0.146  p=0.535
lag_biweekly_t2                R²=0.094  Adj R²=-0.208  p=0.616
roll_same_3days_mean_t2        R²=0.240  Adj R²=-0.140  p=0.510
roll_prev_5days_mean_t1        R²=0.756  Adj R²=0.675  p=0.055
roll_prev_5days_mean_t2        R²=0.461  Adj R²=0.281  p=0.208
lag_same_t1_ratio              R²=0.198  Adj R²=-0.003  p=0.377
lag_same_t2_ratio              R²=0.198  Adj R²=-0.003  p=0.377


In [14]:
single_feature_scan(tue_data, features_incl_t2, 'cumul_20_22', '이상치 포함')


이상치 포함 단순회귀 계수확인
cumul_16_19                    R²=0.183  Adj R²=0.046  p=0.291
lag_same_day_t1                R²=0.119  Adj R²=-0.057  p=0.448
lag_same_day_t2                R²=0.093  Adj R²=-0.089  p=0.507
lag_biweekly_t2                R²=0.145  Adj R²=-0.068  p=0.456
roll_same_3days_mean_t1        R²=0.125  Adj R²=-0.167  p=0.560
roll_prev_5days_mean_t1        R²=0.808  Adj R²=0.760  p=0.015
roll_prev_5days_mean_t2        R²=0.477  Adj R²=0.347  p=0.128
lag_same_t1_ratio              R²=0.356  Adj R²=0.228  p=0.157
lag_same_t2_ratio              R²=0.356  Adj R²=0.228  p=0.157


## 2-4. 스텝와이즈 전진선택


In [15]:
select_t2 = ['roll_prev_5days_mean_t1']  # 단순회귀 R² 최고(0.756)
remain_t2 = [f for f in features_excl_t2 if f not in select_t2]
stepwise_scan(tue_data_not_outlier, select_t2, remain_t2, 'cumul_20_22', 'Step 1 — roll_prev_5days_mean_t1 기준')


【Step 1 — roll_prev_5days_mean_t1 기준】 현재 피처: ['roll_prev_5days_mean_t1']
피처 추가                          R²       Adj R²     p(추가피처)
-----------------------------------------------------------------
cumul_16_19                    R²=0.838  Adj R²=0.676  p=0.421
lag_biweekly_t1                R²=0.786  Adj R²=0.573  p=0.648
lag_biweekly_t2                R²=0.775  Adj R²=0.551  p=0.720
roll_same_3days_mean_t2        R²=0.754  Adj R²=0.263  p=0.968
roll_prev_5days_mean_t2        R²=0.757  Adj R²=0.513  p=0.964
lag_same_t1_ratio              R²=0.829  Adj R²=0.657  p=0.455
lag_same_t2_ratio              R²=0.829  Adj R²=0.657  p=0.455



In [16]:
select_t2 = select_t2 + ['lag_same_t2_ratio']
remain_t2 = [f for f in features_excl_t2 if f not in select_t2]
stepwise_scan(tue_data_not_outlier, select_t2, remain_t2, 'cumul_20_22', 'Step 2 — + lag_same_t2_ratio')


【Step 2 — + lag_same_t2_ratio】 현재 피처: ['roll_prev_5days_mean_t1', 'lag_same_t2_ratio']
피처 추가                          R²       Adj R²     p(추가피처)
-----------------------------------------------------------------
cumul_16_19                    R²=0.975  Adj R²=0.899  p=0.251
lag_biweekly_t1                R²=0.840  Adj R²=0.360  p=0.834
lag_biweekly_t2                R²=0.958  Adj R²=0.831  p=0.330
roll_same_3days_mean_t2        R²=1.000  Adj R²=nan  p=nan
roll_prev_5days_mean_t2        R²=0.834  Adj R²=0.337  p=0.884
lag_same_t1_ratio              R²=0.829  Adj R²=0.657  p=0.597



In [17]:
select_t2 = select_t2 + ['lag_biweekly_t2']
final_model_t2_raw = fit_ols(tue_data_not_outlier, select_t2, 'cumul_20_22')
print("【3피처 모델 - 화요일 cumul_20_22 (표준화 전)】")
print(final_model_t2_raw.summary())


【3피처 모델 - 화요일 cumul_20_22 (표준화 전)】
                            OLS Regression Results                            
Dep. Variable:            cumul_20_22   R-squared:                       0.958
Model:                            OLS   Adj. R-squared:                  0.831
Method:                 Least Squares   F-statistic:                     7.572
Date:                Tue, 14 Jul 2026   Prob (F-statistic):              0.260
Time:                        05:10:45   Log-Likelihood:                -37.592
No. Observations:                   5   AIC:                             83.18
Df Residuals:                       1   BIC:                             81.62
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------

3피처 모델은 Adj R²=0.831로 높아 보이지만 전체 F검정 p=0.260으로 **유의하지 않다** — 표본(5행) 대비 피처(3개)가 과한 상태. 스케일 차이(비율형 피처 vs 수천 단위 연속형 피처)가 계수 해석에 영향을 줄 수 있어, 표준화 후 재적합한 모델을 최종으로 채택한다.


## 2-5. 최종 모델 — cumul_20_22 (표준화, 2피처)


In [18]:
scale_cols = ['cumul_20_22', 'roll_prev_5days_mean_t1']
scaler = StandardScaler()
tue_scaled = tue_data_not_outlier.copy()
tue_scaled[scale_cols] = scaler.fit_transform(tue_data_not_outlier[scale_cols])

final_model_t2 = fit_ols(tue_scaled, ['roll_prev_5days_mean_t1', 'lag_same_t2_ratio'], 'cumul_20_22')
print("【최종 모델 - 화요일 cumul_20_22 (표준화)】")
print(final_model_t2.summary())


【최종 모델 - 화요일 cumul_20_22 (표준화)】
                            OLS Regression Results                            
Dep. Variable:            cumul_20_22   R-squared:                       0.829
Model:                            OLS   Adj. R-squared:                  0.657
Method:                 Least Squares   F-statistic:                     4.835
Date:                Tue, 14 Jul 2026   Prob (F-statistic):              0.171
Time:                        05:10:45   Log-Likelihood:                -1.5470
No. Observations:                   5   AIC:                             9.094
Df Residuals:                       2   BIC:                             7.922
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------

---
## 종합

동일한 스텝와이즈 절차를 T1·T2에 각각 적용한 결과:

| | T1 (cumul_16_19) | T2 (cumul_20_22) |
|---|---|---|
| 최종 피처 | `lag_biweekly_t1` + `prev_sum_g` | `roll_prev_5days_mean_t1` + `lag_same_t2_ratio` (표준화) |
| N | 5 | 5 |
| Adj R² | 0.942 | 0.657 |
| F검정 p-value | **0.0291 (유의)** | **0.171 (기각 실패)** |

인접한 두 시간대(오후/저녁)에 같은 방법론을 적용했는데도 결과가 극단적으로 갈린다 — 표본이 5행뿐인 상태에서는 이런 유의성 여부 자체가 우연에 크게 좌우될 수 있다는 뜻이다. **표본 부족 시 요일별 분리는 설명력은 높아도 실제로는 신뢰 불가능하다**는 결론의 직접적 근거가 된다.
